In [1]:
import duckdb


In [2]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)
con.execute('ALTER TABLE bronze_z0019_2 RENAME TO bronze_z0019')

In [8]:
df = con.execute("""
                 SELECT *
                 FROM (
                 SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) as row_num
                 FROM bronze_z0019
                 WHERE data_ingestao >= '2026-01-01' 
                 ) WHERE row_num = 1
                 """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row_num
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-03-28 15:30:47,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-03-28 15:30:47,1
2,10004,SERRA,BT50,100,500,z0019_2.csv,2026-03-28 15:43:55,1
3,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-28 15:43:55,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-03-28 15:43:55,1


In [13]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row_num'])
df_final = df_final.rename(columns={'NATBR': 'id'})
df_final = df_final.rename(columns={'MAKTX': 'nm_produto'})
df_final = df_final.rename(columns={'WERKS': 'id_categoria'})
df_final = df_final.rename(columns={'MAINS': 'id_fornecedor'})
df_final = df_final.rename(columns={'LABST': 'vl_preco'})
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,500
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,60


In [16]:
df_final.dtypes

id               str
nm_produto       str
id_categoria     str
id_fornecedor    str
vl_preco         str
dtype: object

In [23]:
df2 = df_final
df2.astype(
    {
        'id': 'int64', 
        'nm_produto': 'string', 
        'id_categoria': 'string', 
        'id_fornecedor': 'int64', 
        'vl_preco': 'float64'
        }
    )
#df2.head(10)
df2.dtypes


id               str
nm_produto       str
id_categoria     str
id_fornecedor    str
vl_preco         str
dtype: object